# Run an experiment

Pick a **model**, the **data**, and the **training recipe**, then train. Every run is saved automatically
(`runs/<site>/<run_name>/<timestamp>/`: trained model, config, metrics, predictions) and adds one row to
`results/experiment_log.csv`. Edit only the cells marked **CHOOSE** and run top to bottom.

How each model was trained in its paper: `docs/training_recipes.md` (PDF in `Resources/`). Section 1 can load that
recipe as the starting point.

| Step | What you choose | Code |
|---|---|---|
| 1 Model | architecture, hyperparameters, optionally the paper's recipe | `pvf/models/architectures/`, `pvf/training/paper_recipes.py` |
| 2 Data | site, history / future features, lookback, horizon, split | `pvf/data/` |
| 3 Input adapter | (inspect) how the batch becomes this model's input | `pvf/model_inputs/` |
| 4 Recipe | loss, optimizer, scheduler, epochs, early stopping, transfer... | `pvf/training/` |
| 5-8 | train, diagnose, compare, search | `pvf/experiment.py`, `pvf/evaluation/`, `pvf/search.py` |

In [ ]:
%matplotlib inline
import pandas as pd
import pvf

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

## 0 Machine check (once per machine)

In [ ]:
pvf.check_environment()

environment
  OK    python 3.11.2 (expected 3.11)
  OK    torch 2.14.1+cu130, CUDA runtime 13.0
  OK    GPU: NVIDIA GeForce RTX 3060
  OK    mambapy 1.2.0
  OK    pvlib 0.16.1
  OK    chronos-forecasting 2.3.2
  OK    timesfm 3.0.2
  OK    tirex-ts 1.4.2
data (copy the Data/ folder into the project root)
  OK    4 data files present
  OK    pretrained weights present: ['chronos2', 'timesfm', 'tirex'] (otherwise downloaded on first use)

READY


True

## 1 Model
Every model with its hyperparameters, and how each paper trained it.

In [ ]:
pvf.describe_models()

,model_name,class,trainable,module,hyperparameters (defaults)
0,chronos2,Chronos2,False,foundation_models,"{'weights': 'amazon/chronos-2', 'use_covariates': True, 'batch_size': 256, 'return_quantiles': False}"
1,dt_dann,DigitalTwinTransformerDANN,True,probabilistic,"{'model_size': 64, 'n_heads': 4, 'n_layers': 2, 'feedforward_size': 128, 'dropout': 0.1, 'domain_loss_weight': 0.1, 'use_future_features': True, 'quantile_levels': None}"
2,fusion_itransformer_hdcf,ITransformerHDCFFusion,True,fusion_example,"{'fusion_size': 64, 'n_attention_heads': 4, 'n_transformer_layers': 2, 'state_size': 16, 'dropout': 0.1, 'use_future_features': True, 'scan_backend': 'auto', 'quantile_levels': None}"
3,graph_mamba,GraphMamba,True,graph_mamba,"{'node_feature_names': None, 'model_size': 64, 'temporal_encoder': 'mamba', 'n_temporal_layers': 2, 'graph_layer': 'gat', 'n_graph_layers': 2, 'n_heads': 4, 'adjacency_mode': 'mixed', 'physics_adj..."
4,gru,GRUForecaster,True,baselines,"{'hidden_size': 64, 'n_layers': 2, 'dropout': 0.1, 'bidirectional': False, 'pooling': 'last', 'use_future_features': True, 'quantile_levels': None}"
5,hdcf_mamba,HDCFMamba,True,hdcf_mamba,"{'model_size': 64, 'state_size': 16, 'expansion_factor': 2, 'timeblock_size': 32, 'pool_strides': (2, 4, 8), 'dropout': 0.1, 'use_future_features': True, 'scan_backend': 'auto', 'quantile_levels':..."
6,i_transformer,ITransformer,True,transformers,"{'model_size': 128, 'n_heads': 8, 'n_layers': 3, 'feedforward_size': 256, 'dropout': 0.1, 'use_future_tokens': True, 'quantile_levels': None}"
7,lstm,LSTMForecaster,True,baselines,"{'hidden_size': 64, 'n_layers': 2, 'dropout': 0.1, 'bidirectional': False, 'pooling': 'last', 'use_future_features': True, 'quantile_levels': None}"
8,mamba_temporal,MambaTemporal,True,state_space,"{'model_size': 64, 'n_layers': 3, 'state_size': 16, 'expansion_factor': 2, 'dropout': 0.1, 'use_future_features': True, 'scan_backend': 'auto', 'quantile_levels': None}"
9,mtsa_qrn,MTSAQRN,True,probabilistic,"{'model_size': 64, 'dilations': (1, 2, 4, 8), 'kernel_size': 3, 'n_heads': 4, 'n_graph_heads': 4, 'dropout': 0.1, 'quantile_levels': (0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95), 'use_future_features':..."


In [ ]:
pvf.describe_paper_recipes()

,model_name,source,lookback_hours,optimizer,learning_rate,scheduler,epochs,early_stopping,batch_size,model_hyperparameters,notes
0,chronos2,R12 + R3,504,adamw,0.0010,reduce_on_plateau,100,nrmse / 10,64,{'use_covariates': True},"zero-shot; R3: context saturates near 2000 x 15 min (~3 weeks); R3 LoRA fine-tune (lr 1e-6, 1000 steps, rank 8) was worse on the median - not implemented"
1,dt_dann,R18,24,adam,0.0001,reduce_on_plateau,100,nrmse / 10,64,"{'model_size': 64, 'domain_loss_weight': 1.0}","learning rate, batch and epochs not reported -> Adam 1e-4, consensus otherwise; paper loss weights domain loss 1:1 with the forecast losses; set DataConfig(domain_adaptation_months=[...]) to give ..."
2,fusion_itransformer_hdcf,R6 recipe,96,adam,0.0001,reduce_on_plateau,100,validation_loss / 4,64,"{'fusion_size': 128, 'state_size': 32}",worked example; follows its Mamba branch's paper (R6)
3,graph_mamba,R6 recipe (SET Lab synthesis),96,adam,0.0001,reduce_on_plateau,100,validation_loss / 4,64,{'model_size': 128},no paper; temporal Mamba part follows R6
4,gru,R20 (LSTM settings),240,adamw,0.0010,one_cycle,200,nmae / 30,64,"{'hidden_size': 256, 'n_layers': 2, 'bidirectional': True, 'pooling': 'attention'}",R20 tested LSTM only; same settings used for GRU
5,hdcf_mamba,R6,96,adam,0.0001,reduce_on_plateau,100,validation_loss / 4,64,"{'model_size': 128, 'state_size': 32, 'expansion_factor': 2, 'dropout': 0.1}",plateau-scheduler patience not reported -> 2 (must act before early stop at 4); lookback not stated exactly (inputs up to 720 x 15 min) -> 96 h; paper smooths inputs with a moving average and z-sc...
6,i_transformer,R20,240,adamw,0.0010,one_cycle,200,nmae / 30,64,"{'model_size': 512, 'n_heads': 8, 'n_layers': 3, 'feedforward_size': 1024}",n_layers not reported in R20 -> 3 (original paper: 2-4)
7,lstm,R20,240,adamw,0.0010,one_cycle,200,nmae / 30,64,"{'hidden_size': 256, 'n_layers': 2, 'bidirectional': True, 'pooling': 'attention'}",early stopping on nMAE instead of MAPE (zero PV at night)
8,mamba_temporal,R6 recipe (no own paper),96,adam,0.0001,reduce_on_plateau,100,validation_loss / 4,64,"{'model_size': 128, 'state_size': 32, 'n_layers': 2}",Concat-Mamba ablation root: uses the HDCF-Mamba recipe so the ablation is fair
9,mtsa_qrn,R8,48,adamw,0.0010,reduce_on_plateau,100,nrmse / 10,64,{},no training settings reported -> consensus recipe; pinball loss is automatic; paper decomposes the power history with VMD (not done here); calibrate afterwards with CQR


In [ ]:
# CHOOSE the model
model_name = "time_xer"
use_paper_recipe = True                 # start from the paper's hyperparameters + training recipe + lookback

preset = pvf.paper_recipe(model_name) if use_paper_recipe else None
model_hyperparameters = dict(preset.model_hyperparameters) if preset else {}
model_hyperparameters.update({})        # your changes, e.g. {"model_size": 64}
residual_on_smart_persistence = False   # True: forecast = smart persistence + learned correction
if preset:
    print("paper:", preset.source, "| hyperparameters:", model_hyperparameters)
    for note in preset.notes:
        print("  note:", note)

paper: F03 (original) | hyperparameters: {'patch_length': 24, 'model_size': 256, 'n_layers': 2}
  note: patch 24 = the paper's short-term setting
  note: early-stopping patience not reported -> 3


## 2 Data
Available features: `usable_as_future_feature` = known in advance. `*_oracle` = measured values used as a perfect forecast (an upper bound only).

In [ ]:
pvf.describe_features()

,feature,usable_as_future_feature,description
0,ac_power,False,AC power / rated power
1,irradiance,False,measured irradiance / 1000 W/m2
2,module_temperature,False,(module temperature - 25 C) / 25 C
3,clear_sky_irradiance,True,Ineichen clear-sky GHI / 1000 W/m2
4,clearness_index,False,irradiance / clear-sky irradiance (0 at night)
5,ac_power_yesterday,True,AC power 24 h earlier / rated power
6,hour_of_day_sin,True,sin(2 pi hour/24)
7,hour_of_day_cos,True,cos(2 pi hour/24)
8,day_of_year_sin,True,sin(2 pi day/365)
9,day_of_year_cos,True,cos(2 pi day/365)


In [ ]:
# CHOOSE the data
data_config = pvf.DataConfig(
    site="ssp",                                   # "ssp" (Sirajganj, 2025) | "nist" (Gaithersburg, 2017)
    history_feature_names=["ac_power", "irradiance", "module_temperature", "clear_sky_irradiance",
                           "hour_of_day_sin", "hour_of_day_cos", "day_of_year_sin", "day_of_year_cos"],
    future_feature_names=["clear_sky_irradiance", "ac_power_yesterday",
                          "hour_of_day_sin", "hour_of_day_cos", "day_of_year_sin", "day_of_year_cos"],
    lookback_steps=preset.lookback_steps("30min") if preset else 128,   # x 30 min of history
    horizon_steps=48,                             # x 30 min ahead (48 = 24 h)
    split=pvf.SplitConfig(mode="blocked"),        # weekly blocks: 4 train / 1 validation / 1 test, every season
)
data_bundle = pvf.build_data_bundle(data_config)
pvf.describe_data_bundle(data_bundle)

,split,samples,first_origin,last_origin,months
0,train,9561,2025-01-04,2025-12-29,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]"
1,validation,2109,2025-01-28,2025-12-30,"[1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12]"
2,test,2182,2025-02-04,2025-12-01,"[2, 3, 5, 6, 7, 9, 10, 11, 12]"


In [ ]:
model = pvf.build_model(model_name, data_bundle.spec, residual_on_smart_persistence, **model_hyperparameters)
model

TimeXer(
  (patch_embedding): Linear(in_features=24, out_features=256, bias=True)
  (history_variable_embedding): Linear(in_features=192, out_features=256, bias=True)
  (future_variable_embedding): Linear(in_features=48, out_features=256, bias=True)
  (layers): ModuleList(
    (0-1): 2 x TimeXerLayer(
      (patch_self_attention): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (global_cross_attention): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (norm_self): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm_cross): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm_feedforward): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (feedforward): Sequential(
        (0): Linear(in_features=256, out_features=256, bias=True)
        (1): GELU(approx

## 3 Input adapter
How the standard batch from the data pipeline becomes this model's input, step by step.

In [ ]:
batch = pvf.sample_batch(data_bundle)
print(model.input_adapter)
model.input_adapter.trace(batch)

In [ ]:
model.input_adapter.plot(batch);

## 4 Training recipe

In [ ]:
print("losses:", list(pvf.LOSSES), "| optimizers:", list(pvf.OPTIMIZERS), "| schedulers:", list(pvf.SCHEDULERS))
recipe = preset.recipe if preset else pvf.TrainingRecipe()

# CHOOSE: change any field (all fields: pvf.TrainingRecipe?)
recipe.n_epochs = 100
# recipe.loss = "huber";              recipe.loss_options = {"delta": 0.05}
# recipe.optimizer = "adamw";         recipe.learning_rate = 1e-3;  recipe.weight_decay = 1e-4
# recipe.scheduler = "one_cycle";     recipe.scheduler_options = {}
# recipe.warmup_epochs = 3
# recipe.early_stopping_metric = "nrmse";  recipe.early_stopping_patience = 15   # None = train all epochs
# recipe.batch_size = 32;             recipe.gradient_accumulation_steps = 2
# recipe.train_fraction = 0.2                                                    # data-efficiency curve
# recipe.initialize_from_run = r"runs/nist/hdcf_mamba_nist/<timestamp>"          # transfer from another run
# recipe.freeze_parameter_prefixes = ["encoder."]
run_name = f"{model_name}_{data_config.site}"      # label used in comparisons
recipe

## 5 Train and score
Saves the trained model and all results; prints standard metrics per split.

In [ ]:
experiment_config = pvf.ExperimentConfig(model_name=model_name, model_hyperparameters=model_hyperparameters,
                                         residual_on_smart_persistence=residual_on_smart_persistence,
                                         data=data_config, recipe=recipe, run_name=run_name)
result = pvf.run_experiment(experiment_config, data_bundle=data_bundle, model=model)
result.summary()

## 6 Diagnose this run

In [ ]:
pvf.plot_training_history(result.run_dir);

In [ ]:
pvf.plot_forecast_examples(result.run_dir, split="test", n_examples=3);

In [ ]:
pvf.plot_per_horizon([result.run_dir], metric="nrmse_pct", split="test");

In [ ]:
pvf.plot_error_by_hour([result.run_dir], split="test");

## 7 Compare runs
Latest run per run name, best first. Column meanings: `pvf.evaluation.metrics.METRIC_DESCRIPTIONS`.

In [ ]:
comparison = pvf.compare_runs(site=data_config.site, split="test")
comparison

In [ ]:
best_runs = comparison.run_dir[:5]
pvf.plot_per_horizon(best_runs, metric="nrmse_pct", split="test");
pvf.plot_error_by_hour(best_runs, split="test");

## 8 Hyperparameter search (optional)
The papers tuned with Optuna, grid search or Nixtla. Keys are `model.<hyperparameter>` or `recipe.<field>`; every
trial is a saved run, ranked on the **validation** split.

In [ ]:
# CHOOSE the search space, then set run_search = True
run_search = False
if run_search:
    search_results = pvf.search_hyperparameters(experiment_config, {
        "model.model_size": [64, 128],
        "recipe.learning_rate": [1e-4, 3e-4, 1e-3],
    }, n_trials=6, data_bundle=data_bundle)
    display(search_results)

## 9 Fusing models
`pvf/models/architectures/fusion_example.py` fuses iTransformer's variable encoder with HDCF-Mamba's temporal encoder
through a learned gate (4-step recipe in its docstring). Try it with `model_name = "fusion_itransformer_hdcf"`.
To build your own: copy that file, swap in two other encoders, register it under a new name.

## 10 Reload a saved model

In [ ]:
trained_model = pvf.load_trained_model(result.run_dir)        # ready to call on any ForecastBatch